# Brazilian Chamber Voting Networks

## Theory, intuition, research design, methodology, and limitations

This document explains the ideas behind the project in a more visual way.

The core idea is:

> A roll-call vote is one observation. A sequence of roll-call votes becomes a behavioral profile. Similar profiles can be compared statistically, and those similarities can be represented as a network.

```mermaid
flowchart LR
    A[Roll-call votes] --> B[Vote encoding]
    B --> C[Voting profiles]
    C --> D[Similarity matrix]
    D --> E[Network construction]
    E --> F[Network metrics]
    F --> G[Political interpretation]
```

The most important interpretation rule is:

$$
\boxed{
\text{observed voting similarity}
\neq
\text{causality, ideology, or political influence}
}
$$

The network summarizes **how similarly actors voted in the observed data**. Any stronger political interpretation requires additional assumptions or external evidence.

---

# 1. Research question

The project asks a relational question:

> **Which parties or deputies tend to vote similarly, and what kind of network structure emerges from those similarities?**

Instead of analyzing each deputy or party in isolation, we study the pattern of relationships among them.

This lets us ask questions such as:

- Which parties frequently vote in similar ways?
- Which actors behave as bridges between otherwise separate groups?
- Do formal party boundaries coincide with behavioral communities?
- Does the network structure change across legislatures or governments?
- Are the results stable when we change the correlation threshold?

---

# 2. Data source and basic observation

The project uses roll-call voting data from the Brazilian Chamber of Deputies Open Data system.

A simplified raw record looks like this:

| Vote ID | Deputy | Party | Vote | Date |
|---|---|---|---|---|
| V1 | Deputy A | Party X | Yes | 2020-03-10 |
| V1 | Deputy B | Party X | No | 2020-03-10 |
| V1 | Deputy C | Party Y | Yes | 2020-03-10 |

A single row says little by itself. The information becomes useful when we organize many votes into a profile.

```mermaid
flowchart TD
    A[One deputy] --> B[Vote 1]
    A --> C[Vote 2]
    A --> D[Vote 3]
    A --> E[...]
    A --> F[Vote T]

    B --> G[Behavioral profile]
    C --> G
    D --> G
    E --> G
    F --> G
```

---

# 3. Encoding categorical votes

The raw vote is categorical, so the first step is to create a numerical representation.

A simple encoding is:

$$
x_{it}=
\begin{cases}
+1, & \text{if deputy } i \text{ votes Yes in vote } t,\\
-1, & \text{if deputy } i \text{ votes No in vote } t,\\
0, & \text{for selected non-binary categories.}
\end{cases}
$$

For example:

| Recorded vote | Numerical value |
|---|---:|
| Yes | +1 |
| No | -1 |
| Abstention | 0 |
| Obstruction | 0 |

### Small example

Suppose a deputy votes:

```text
Yes, Yes, No, Abstention, Yes
```

The numerical profile becomes:

$$
\mathbf{x}_i=(1,1,-1,0,1).
$$

### Intuition

The encoding creates a signed signal:

```text
support      neutral/non-directional      opposition
   +1  <-----------  0  ----------->  -1
```

The zero category is a modeling choice. Abstention and obstruction may have different political meanings, so treating both as zero is convenient but not neutral from a substantive point of view.

That is why alternative encodings should be tested in robustness analyses.

---

# 4. Voting profiles

For deputy $i$, define the voting profile

$$
\mathbf{x}_i=
(x_{i1},x_{i2},\ldots,x_{iT}),
$$

where $T$ is the number of roll-call votes.

### Example

Consider two deputies:

$$
\mathbf{x}_A=(1,1,-1,1,0)
$$

and

$$
\mathbf{x}_B=(1,1,-1,1,1).
$$

Their profiles are very similar.

Now compare them with

$$
\mathbf{x}_C=(-1,-1,1,-1,0).
$$

Deputy C usually moves in the opposite direction.

Visually:

```text
Vote        1   2   3   4   5
--------------------------------
Deputy A    +   +   -   +   0
Deputy B    +   +   -   +   +
Deputy C    -   -   +   -   0
```

The statistical problem is to transform this intuitive idea of "similar voting behavior" into a reproducible measure.

---

# 5. Aggregating deputies into party profiles

When the analysis is performed at the party level, individual votes are aggregated within each roll call.

Let $P_{pt}$ be the set of observed deputies from party $p$ in vote $t$.

A simple party score is

$$
S_{pt}
=
\sum_{i\in P_{pt}}x_{it}.
$$

With the encoding $+1$ for Yes and $-1$ for No,

$$
S_{pt}
=
N^{\text{Yes}}_{pt}
-
N^{\text{No}}_{pt}.
$$

## Example

Suppose Party A has the following observed votes:

```text
Yes, Yes, Yes, No, Abstention
```

Then

$$
S_{At}
=
1+1+1-1+0
=
2.
$$

A positive value means the observed party vote leans toward Yes.

A negative value means it leans toward No.

A value close to zero means the observed party vote is balanced or dominated by zero-coded categories.

```mermaid
flowchart LR
    A[Individual votes] --> B[+1]
    A --> C[-1]
    A --> D[0]
    B --> E[Sum]
    C --> E
    D --> E
    E --> F[Party score S_pt]
```

---

# 6. Why normalize by party size?

The raw score $S_{pt}$ has an important limitation: larger parties can naturally produce larger absolute values.

Imagine:

```text
Party A: 40 Yes, 10 No
Party B:  4 Yes,  1 No
```

The raw scores are

$$
S_A=40-10=30,
$$

$$
S_B=4-1=3.
$$

But the internal proportions are identical.

To reduce this size effect, define

$$
N_{pt}=|P_{pt}|
$$

and

$$
M_{pt}
=
\frac{S_{pt}}{N_{pt}}.
$$

For the example:

$$
M_A=\frac{30}{50}=0.6,
$$

$$
M_B=\frac{3}{5}=0.6.
$$

Now the two parties receive the same normalized stance.

### Interpretation

| Measure | Main meaning |
|---|---|
| $S_{pt}$ | Aggregate voting balance |
| $M_{pt}$ | Average observed stance |

Neither is universally better. They answer different questions.

---

# 7. Measuring similarity with Pearson correlation

After building one voting vector per party, we compare parties pairwise.

For party $p$,

$$
\mathbf{s}_p=
(S_{p1},S_{p2},\ldots,S_{pT}).
$$

For parties $p$ and $q$, Pearson correlation is

$$
\rho_{pq}
=
\frac{
\operatorname{Cov}(S_p,S_q)
}{
\sigma_p\sigma_q
}.
$$

The coefficient satisfies

$$
-1\leq\rho_{pq}\leq1.
$$

### Intuition

```text
rho ≈ +1   move together
rho ≈  0   weak linear association
rho ≈ -1   move in opposite directions
```

### Tiny example

Suppose:

$$
\mathbf{s}_A=(2,3,-1,4)
$$

and

$$
\mathbf{s}_B=(1,2,-1,3).
$$

The two profiles rise and fall together, so the correlation should be strongly positive.

Now suppose

$$
\mathbf{s}_C=(-2,-3,1,-4).
$$

Party C moves almost exactly in the opposite direction, so its correlation with A should be close to $-1$.

The correct interpretation is

$$
\boxed{
\rho_{pq}
=
\text{similarity in observed voting profiles}
}
$$

not

$$
\boxed{
\rho_{pq}
=
\text{ideological distance or causal influence}
}
$$

---

# 8. Missing votes and common overlap

Two parties may not be observed in exactly the same set of roll calls.

Define

$$
n_{pq}
=
\left|
\left\{
t:
S_{pt}\text{ and }S_{qt}\text{ are both observed}
\right\}
\right|.
$$

This is the number of votes that both parties have in common.

### Why it matters

A correlation of $0.80$ based on 8 common votes is not as convincing as a correlation of $0.80$ based on 500 common votes.

So the pipeline should require

$$
n_{pq}\geq n_{\min}.
$$

A useful result table should therefore show both:

| Pair | Correlation | Common votes |
|---|---:|---:|
| A-B | 0.82 | 420 |
| A-C | 0.80 | 9 |

The correlations are similar, but the statistical support is not.

---

# 9. From a correlation matrix to a network

The pairwise correlations form a matrix

$$
R=[\rho_{pq}].
$$

For three parties:

|  | A | B | C |
|---|---:|---:|---:|
| A | 1.00 | 0.82 | 0.20 |
| B | 0.82 | 1.00 | 0.55 |
| C | 0.20 | 0.55 | 1.00 |

To create a graph, we define a threshold $\tau$.

$$
A_{pq}(\tau)
=
\begin{cases}
1, & \rho_{pq}>\tau,\\
0, & \text{otherwise}.
\end{cases}
$$

Suppose

$$
\tau=0.60.
$$

Then:

```text
A-B survives because 0.82 > 0.60
A-C disappears because 0.20 < 0.60
B-C disappears because 0.55 < 0.60
```

The graph becomes:

```mermaid
graph LR
    A((Party A)) ---|0.82| B((Party B))
    C((Party C))
```

The weighted network can be written as

$$
G=(V,E,W),
$$

where:

- $V$ is the set of parties or deputies;
- $E$ is the set of retained relationships;
- $W$ contains edge weights.

---

# 10. The threshold is part of the model

A threshold is not just a plotting preference.

If $\tau$ is low:

$$
\tau\downarrow
\quad\Rightarrow\quad
|E|\uparrow.
$$

The graph becomes denser.

If $\tau$ is high:

$$
\tau\uparrow
\quad\Rightarrow\quad
|E|\downarrow.
$$

The graph becomes sparser.

```mermaid
flowchart LR
    A[Low threshold] --> B[Many edges]
    B --> C[Dense network]

    D[High threshold] --> E[Few edges]
    E --> F[Sparse network]
```

Instead of using one arbitrary value, the project should compare several values:

$$
\tau\in
\{0.50,0.60,0.70,0.75,0.80,0.90\}.
$$

For every threshold, compare:

- number of edges;
- density;
- connected components;
- average degree;
- centrality measures;
- community structure.

This is a sensitivity analysis.

---

# 11. Fixed threshold versus backbone extraction

A fixed threshold is transparent:

```text
keep edge if correlation > tau
```

But it is not the only possible strategy.

```mermaid
flowchart TD
    A[Similarity matrix] --> B{How should weak edges be removed?}
    B --> C[Fixed threshold]
    B --> D[Keep weighted graph]
    B --> E[Backbone extraction]
```

Backbone methods try to preserve structurally meaningful relationships while removing weak or statistically uninformative edges.

For this project, a strong methodological comparison is:

$$
\boxed{
\text{fixed threshold}
\quad\text{vs.}\quad
\text{weighted network}
\quad\text{vs.}\quad
\text{backbone extraction}
}
$$

The goal is not to choose the prettiest graph. The goal is to see whether the substantive conclusion survives different reasonable construction rules.

---

# 12. What node size means

In party graphs, node size can represent the number of observed unique deputies:

$$
n_p
=
\left|
\left\{
i:
i\text{ is observed in party }p
\right\}
\right|.
$$

A plotting rule may be

$$
\text{size}_p=c\,n_p,
$$

where $c$ is only a visual scaling factor.

### Example

```text
Party A: 60 deputies  -> large node
Party B: 15 deputies  -> small node
```

This does **not** mean Party A is more central in the network.

Node size and network centrality are different concepts.

---

# 13. What edge width means

For retained positive correlations, one possible plotting rule is

$$
\text{width}_{pq}
=
c_w\rho_{pq}.
$$

If:

```text
A-B correlation = 0.90
A-C correlation = 0.65
```

then the A-B edge should appear thicker.

The multiplier $c_w$ is purely graphical.

---

# 14. ForceAtlas2: what it does

ForceAtlas2 is a **layout algorithm**.

Its purpose is to choose positions for the nodes so that the network is easier to visualize.

Conceptually:

```mermaid
flowchart LR
    A[Connected nodes] -->|attraction| C[Move closer]
    B[All nodes] -->|repulsion| D[Move apart]
    C --> E[Iterative layout]
    D --> E
    E --> F[Readable network map]
```

A very simplified mental model is:

```text
edge attraction  <----->  node repulsion
```

The algorithm repeatedly adjusts positions until the drawing becomes relatively stable.

### Critical point

ForceAtlas2 does not estimate ideology.

$$
\boxed{
\text{ForceAtlas2 coordinates}
\neq
\text{ideological coordinates}
}
$$

Also:

$$
\boxed{
\text{distance in the drawing}
\neq
\text{causal or ideological distance}
}
$$

The layout is useful for visualization. It should not be interpreted as a statistical estimator.

---

# 15. Similarity networks versus ideal-point models

These are related but different approaches.

## Similarity network

Question:

> Who behaves similarly in the observed voting data?

Primary object:

$$
\rho_{pq}.
$$

Output:

```text
weighted graph
```

## Ideal-point model

Question:

> What latent political position could explain observed voting choices?

A simplified representation is

$$
P(y_{it}=1)
=
f(\theta_i,\alpha_t,\beta_t),
$$

where:

- $\theta_i$ is a latent position for legislator $i$;
- $\alpha_t$ and $\beta_t$ characterize roll call $t$;
- $f(\cdot)$ is a probabilistic voting model.

```mermaid
flowchart LR
    A[Roll-call votes] --> B[Similarity approach]
    A --> C[Ideal-point approach]

    B --> D[Pairwise behavioral similarity]
    D --> E[Network]

    C --> F[Latent political dimension]
    F --> G[Estimated ideal points]
```

The two methods can complement each other, but they should not be interpreted as equivalent.

---

# 16. Network density

For an undirected simple graph with $n$ nodes and $m$ edges,

$$
D=
\frac{2m}{n(n-1)}.
$$

The denominator

$$
\frac{n(n-1)}{2}
$$

is the maximum possible number of undirected edges.

Therefore,

$$
0\leq D\leq1.
$$

### Example

Suppose:

```text
n = 4 nodes
m = 3 edges
```

Then

$$
D=
\frac{2(3)}{4(3)}
=
0.5.
$$

Half of all possible edges are present.

---

# 17. Degree and weighted degree

The degree of node $i$ is

$$
k_i=
\sum_j A_{ij}.
$$

It counts how many neighbors the node has.

Weighted degree, often called strength, is

$$
s_i=
\sum_j w_{ij}.
$$

### Example

```mermaid
graph LR
    A((A)) ---|0.9| B((B))
    A ---|0.7| C((C))
    A ---|0.6| D((D))
```

For node A:

$$
k_A=3
$$

and

$$
s_A=0.9+0.7+0.6=2.2.
$$

Degree measures the number of connections.

Strength measures the total weight of those connections.

---

# 18. Betweenness centrality

Betweenness centrality measures how often a node lies on shortest paths between other nodes.

$$
C_B(v)
=
\sum_{s\neq v\neq t}
\frac{
\sigma_{st}(v)
}{
\sigma_{st}
}.
$$

### Intuition

Consider:

```mermaid
graph LR
    A((A)) --- B((B))
    B --- C((C))
    C --- D((D))
    C --- E((E))
```

Node C connects the left side to nodes D and E.

Many shortest paths pass through C, so C can have high betweenness.

The safe interpretation is:

> C is a structural bridge in the constructed similarity network.

That is more defensible than saying:

> C is politically influential.

---

# 19. Closeness centrality

A common form is

$$
C_C(i)
=
\frac{n-1}{
\sum_{j\neq i}d(i,j)
},
$$

where $d(i,j)$ is the shortest-path distance between nodes $i$ and $j$.

### Intuition

A node has high closeness if it can reach many other nodes through relatively short paths.

Again, this describes the graph produced by our modeling choices. It is not a direct measure of political power.

---

# 20. Community detection and Louvain

Community detection tries to find groups that are more densely connected internally than externally.

A simplified picture:

```mermaid
graph LR
    A1((A1)) --- A2((A2))
    A1 --- A3((A3))
    A2 --- A3

    B1((B1)) --- B2((B2))
    B1 --- B3((B3))
    B2 --- B3

    A3 -. weak link .- B1
```

The Louvain algorithm searches for a partition with high modularity.

A common modularity formula is

$$
Q
=
\frac{1}{2m}
\sum_{i,j}
\left[
A_{ij}
-
\frac{k_i k_j}{2m}
\right]
\delta(c_i,c_j).
$$

The intuition is:

```text
observed within-community connectivity
minus
connectivity expected under a null model
```

If the observed internal connection is much stronger than expected, modularity increases.

---

# 21. Party labels are not network communities

A formal political party is an institutional category.

A network community is an algorithmic group induced by the graph.

Therefore,

$$
\boxed{
\text{party}
\neq
\text{network community}
}
$$

This difference is analytically useful.

Possible questions:

- Do members of one party stay in the same behavioral community?
- Do multiple parties merge into one voting community?
- Does one formal party split across several communities?

---

# 22. Political-orientation colors

Colors such as:

```text
red       = left
pink      = moderate left
blue      = right
lightblue = center-right
```

are **external metadata**.

The network itself does not infer these labels.

```mermaid
flowchart LR
    A[Network model] --> B[Voting similarity]
    C[External political classification] --> D[Node color]
    B --> E[Final visualization]
    D --> E
```

If political-orientation labels are used, document:

- source;
- date or period;
- classification rule;
- uncertainty or disagreement;
- unclassified cases.

The visual color should not be presented as an output of Pearson correlation, Louvain, ForceAtlas2, or NetworkX.

---

# 23. Temporal analysis

The project can construct different networks by:

- legislature;
- year;
- government;
- political period.

Write them as

$$
G^{(1)},G^{(2)},\ldots,G^{(K)}.
$$

Then compare:

$$
D^{(k)},
\qquad
\bar{k}^{(k)},
\qquad
Q^{(k)},
\qquad
C^{(k)}.
$$

But observed change across periods can come from several sources:

```mermaid
flowchart TD
    A[Observed network change] --> B[Behavioral change]
    A --> C[Party composition]
    A --> D[Legislative agenda]
    A --> E[Participation]
    A --> F[Data availability]
```

So a different graph in two governments does not, by itself, prove that the government caused the difference.

---

# 24. Party switching and historical harmonization

Brazilian parties change over time through renaming, mergers, incorporation, dissolution, and switching by deputies.

For long-run analysis, it is useful to preserve two fields:

```text
party_raw
party_harmonized
```

Example:

```text
PMDB  -> MDB
DEM   -> UNIÃO
PFL   -> UNIÃO
```

The harmonized label helps comparison, but the raw historical identity should remain available.

Otherwise, the preprocessing step can erase politically meaningful distinctions.

---

# 25. Full research design

A defensible workflow is:

```mermaid
flowchart TD
    A[Official roll-call records]
    B[Data validation]
    C[Historical party harmonization]
    D[Vote encoding]
    E[Individual voting profiles]
    F[Party aggregation]
    G[Minimum-overlap filtering]
    H[Similarity matrix]
    I[Network construction]
    J[Threshold / backbone robustness]
    K[Network metrics]
    L[Community detection]
    M[Temporal comparison]
    N[Substantive interpretation]

    A --> B --> C --> D --> E --> F --> G --> H --> I --> J --> K --> L --> M --> N
```

Every arrow introduces assumptions.

That is why the methodology should be explicit and reproducible.

---

# 26. Robustness checklist

A strong version of the project should test several modeling choices.

| Component | Main robustness question |
|---|---|
| Vote encoding | What happens if abstention or obstruction is encoded differently? |
| Party aggregation | Do raw and normalized party scores tell the same story? |
| Similarity | Are results similar with Pearson, Spearman, cosine, or agreement rate? |
| Overlap | Do results change when $n_{\min}$ changes? |
| Threshold | Are conclusions stable across several $\tau$ values? |
| Network extraction | Do weighted, thresholded, and backbone networks agree? |
| Communities | Do Louvain and Leiden produce similar structure? |
| Time window | Are conclusions stable across different temporal partitions? |

---

# 27. Main limitations

## 27.1 Correlation is not causality

Two parties can vote similarly because of many mechanisms:

```text
shared preferences
coalition structure
party discipline
agenda composition
leadership
institutional incentives
strategic behavior
```

The network alone cannot identify which mechanism caused the similarity.

---

## 27.2 Similarity is not ideology

A voting-similarity network reflects observed behavior.

It does not automatically estimate ideological position.

To claim ideological positioning, we would need a validated mapping or a latent-variable model designed for that purpose.

---

## 27.3 The legislative agenda matters

Suppose Period A contains mostly economic votes and Period B contains mostly procedural votes.

Even if politicians themselves do not change much, the observed similarity structure can change because the **questions being voted on changed**.

This is a major limitation of temporal comparisons.

---

## 27.4 Missingness may be informative

An absent deputy is not the same thing as a neutral deputy.

```text
absence != abstention != obstruction != vote = 0
```

If missing observations are systematically related to political behavior, the estimated profiles may be affected.

---

## 27.5 Party-size effects

Raw party sums can make large parties look more extreme in magnitude simply because more deputies contribute to the sum.

This motivates the normalized score

$$
M_{pt}=
\frac{S_{pt}}{N_{pt}}.
$$

---

## 27.6 Threshold dependence

Changing $\tau$ changes the graph.

```mermaid
flowchart LR
    A[Change threshold] --> B[Change edges]
    B --> C[Change density]
    B --> D[Change centrality]
    B --> E[Change communities]
    B --> F[Change components]
```

So one graph at one threshold is not enough for a strong methodological conclusion.

---

## 27.7 Different pairs may have different overlap

Two party pairs can have the same correlation but very different numbers of common votes.

That means equal correlations can have very different reliability.

---

## 27.8 Multiple comparisons

With $P$ parties, the number of pairs is

$$
\binom{P}{2}
=
\frac{P(P-1)}{2}.
$$

For example, with 20 parties:

$$
\binom{20}{2}
=
190.
$$

That is already 190 pairwise relationships.

As the number of parties grows, some strong correlations may appear by chance, especially with small overlap.

---

## 27.9 Community detection is model-dependent

Louvain is heuristic.

The final partition can depend on:

- threshold;
- edge weights;
- resolution;
- initialization;
- algorithm.

Communities should therefore be treated as structural summaries, not absolute political truths.

---

## 27.10 ForceAtlas2 is not an estimator

Different ForceAtlas2 parameters can produce different drawings of the same graph.

Rotation, translation, scale, and exact coordinates have no direct political meaning.

---

## 27.11 Centrality is not power

A high-centrality node is structurally central under the chosen graph definition.

That does not automatically imply:

```text
greater political power
greater ideological moderation
greater institutional influence
greater legislative importance
```

Those claims require separate evidence.

---

# 28. What the project can claim

With a validated pipeline, statements like these are defensible:

> Parties A and B displayed high similarity in their observed voting profiles during the selected period.

> The network constructed with threshold $\tau$ contained a given number of edges and a given density.

> Party A occupied a high-betweenness position in the constructed similarity network.

> Louvain grouped a set of parties into the same network community under the specified construction rule.

> The main structural pattern remained stable across the tested threshold range.

These statements are tied directly to observable data and explicit modeling choices.

---

# 29. What the project should not claim from the graph alone

The network alone does not prove that:

- one party caused another to vote in a certain way;
- spatial proximity equals ideological proximity;
- ForceAtlas2 coordinates are latent ideological positions;
- a central node has more institutional power;
- a Louvain community is an official coalition;
- a correlation edge proves strategic coordination;
- node color was inferred from the network.

A useful mental rule is:

```text
DESCRIBE what the model measures.
TEST what depends on modeling choices.
DO NOT over-interpret what the graph cannot identify.
```

---

# 30. Methodological extensions

## Deputy-level network

Instead of aggregating to parties, define a network directly on deputies:

$$
G_D=(V_D,E_D,W_D).
$$

This preserves within-party heterogeneity.

---

## Temporal or multilayer networks

Represent each time period as one layer:

$$
G^{(t)}.
$$

This makes network evolution an explicit object of study.

---

## Compare with ideal-point models

A strong extension is to compare:

```text
network similarity
vs.
latent ideal-point distance
```

If both tell similar stories, that is informative.

If they disagree, that is also informative because they are measuring different objects.

---

## Null models

Observed network metrics can be compared with randomized networks.

The question becomes:

> Is the observed structure stronger than what we would expect under a reasonable random benchmark?

That moves the project from description toward formal structural inference.

---

# 31. Reproducibility checklist

Every final graph should record at least:

```text
Data period
Legislature / government
Vote encoding
Party harmonization rule
Aggregation rule
Minimum common votes
Similarity metric
Correlation threshold
Signed or absolute correlation rule
Community algorithm
Community resolution
Layout algorithm
Layout parameters
Random seed
```

This makes the figure reproducible instead of merely visually attractive.

---

# 32. Final intuition

The entire methodology can be summarized as:

```mermaid
flowchart LR
    A[Individual votes]
    B[Numerical signals]
    C[Party / deputy profiles]
    D[Pairwise similarity]
    E[Network]
    F[Graph structure]
    G[Political interpretation]

    A --> B --> C --> D --> E --> F --> G
```

Mathematically:

$$
x_{it}
\longrightarrow
S_{pt}
\longrightarrow
\rho_{pq}
\longrightarrow
A_{pq}(\tau)
\longrightarrow
G
\longrightarrow
\text{network statistics}.
$$

The farther we move from the original vote toward political interpretation, the more assumptions we introduce.

That is the central methodological principle of the project.

---

# References

1. **Brito, A. C. M., Silva, F. N., & Amancio, D. R. (2020).** A complex network approach to political analysis: Application to the Brazilian Chamber of Deputies. *PLOS ONE, 15*(3), e0229928. DOI: `10.1371/journal.pone.0229928`.

2. **Blondel, V. D., Guillaume, J.-L., Lambiotte, R., & Lefebvre, E. (2008).** Fast unfolding of communities in large networks. *Journal of Statistical Mechanics: Theory and Experiment*, P10008. DOI: `10.1088/1742-5468/2008/10/P10008`.

3. **Clinton, J., Jackman, S., & Rivers, D. (2004).** The Statistical Analysis of Roll Call Data. *American Political Science Review, 98*(2), 355-370. DOI: `10.1017/S0003055404001194`.

4. **Figueiredo, A. C., & Limongi, F. (2000).** Presidential Power, Legislative Organization, and Party Behavior in Brazil. *Comparative Politics, 32*(2), 151-170. DOI: `10.2307/422395`.

5. **Jacomy, M., Venturini, T., Heymann, S., & Bastian, M. (2014).** ForceAtlas2, a Continuous Graph Layout Algorithm for Handy Network Visualization Designed for the Gephi Software. *PLOS ONE, 9*(6), e98679. DOI: `10.1371/journal.pone.0098679`.

6. **Poole, K. T., & Rosenthal, H. (1985).** A Spatial Model for Legislative Roll Call Analysis. *American Journal of Political Science, 29*(2), 357-384. DOI: `10.2307/2111172`.

7. **Waugh, A. S., Pei, L., Fowler, J. H., Mucha, P. J., & Porter, M. A.** Party Polarization in Congress: A Network Science Approach.

8. **Masuda, N., Boyd, Z. M., Garlaschelli, D., & Mucha, P. J.** Correlation networks: interdisciplinary approaches beyond thresholding.

9. **Câmara dos Deputados.** Dados Abertos da Câmara dos Deputados: voting and individual parliamentary vote documentation.

10. **Complex networks applied to political analysis: Group voting behavior in the Brazilian congress.** *PLOS ONE* (2025). DOI: `10.1371/journal.pone.0319643`.

---

## Citation note

Political-orientation labels, coalition classifications, and government/opposition metadata should be cited independently from the network methodology.

They are external classifications. They are not inferred by Pearson correlation, ForceAtlas2, Louvain, or NetworkX.
